In [1]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, ValidationError
from google.genai import errors as genai_errors

In [2]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY not found. Check that your .env file exists "
        "and contains GEMINI_API_KEY=your_key_here"
    )

client = genai.Client(api_key=api_key)

In [3]:
class ClientEnquiry(BaseModel):
    client_name: str
    company: str
    industry: str
    requirement: list[str]
    urgency: str
    missing_information: list[str]

In [18]:
SYSTEM_PROMPT = """Extract fields from the client's enquiry.
Rules:
- Never guess. If a value isn't stated, leave it as an empty string or empty list.
- List every field left empty in missing_information.
"""

In [30]:
def prompt_passing(prompt):
    # 1. Empty input check
    if not prompt or not prompt.strip():
        return {"error": "empty_input", "message": "No enquiry text was provided."}

    try:
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.2,
                max_output_tokens=1000,
                top_p=0.9,
                response_mime_type="application/json",
                response_schema=ClientEnquiry,
                thinking_config=types.ThinkingConfig(thinking_level="low"),
            ),
        )

    # 2. API failure (network, auth, rate limit, server errors)
    except genai_errors.APIError as e:
        return {"error": "api_failure", "message": f"API call failed: {e}"}
    except Exception as e:
        return {"error": "unexpected_failure", "message": f"Unexpected error: {e}"}

    # 3. Invalid/blocked response check (no candidates, safety block, etc.)
    if not response.candidates or not response.text:
        return {
            "error": "invalid_response",
            "message": "Model returned no usable content (possibly blocked or empty).",
        }

    # 4. Validate that the response actually matches our schema
    try:
        parsed = ClientEnquiry.model_validate_json(response.text)
        return parsed.model_dump()
    except ValidationError as e:
        return {
            "error": "invalid_response",
            "message": f"Response did not match expected schema: {e}",
            "raw_text": response.text,
        }
    except json.JSONDecodeError as e:
        return {
            "error": "invalid_response",
            "message": f"Response was not valid JSON: {e}",
            "raw_text": response.text,
        }

In [31]:
prompt = "Hi, I'm Jason from GreenTech Energy. We're looking for an AI chatbot, an internal document search assistant, and an analytics dashboard powered by machine learning. We'd like to kick things off within three months."

In [32]:
result = prompt_passing(prompt)

In [33]:
result

{'client_name': 'Jason',
 'company': 'GreenTech Energy',
 'industry': '',
 'requirement': ['AI chatbot',
  'internal document search assistant',
  'analytics dashboard powered by machine learning'],
 'urgency': 'within three months',
 'missing_information': ['industry']}

In [34]:
prompt = "Looking for AI to predict sales."
result = prompt_passing(prompt)
result

{'client_name': '',
 'company': '',
 'industry': '',
 'requirement': ['AI to predict sales'],
 'urgency': '',
 'missing_information': ['client_name', 'company', 'industry', 'urgency']}

In [35]:
with open("test_inputs.json", "r", encoding="utf-8") as file:
    enquiries = json.load(file)

for enquiry in enquiries:
    print(enquiry["id"])
    print(enquiry["category"])
    print(enquiry["client_enquiry"])

    result = prompt_passing(enquiry["client_enquiry"])

    if isinstance(result, dict) and "error" in result:
        print(f"{result['error']}: {result['message']}")
    else:
        print(result)

    print("-" * 50)

1
Complete Enquiry
Hello, my name is Sarah Johnson, and I work at BrightMart Retail. We are looking for an AI chatbot that can answer customer queries on our website and integrate with Shopify. We'd like to launch it within the next two weeks for our marketing campaign.
{'client_name': 'Sarah Johnson', 'company': 'BrightMart Retail', 'industry': 'Retail', 'requirement': ['AI chatbot for customer queries', 'Shopify integration'], 'urgency': 'Within the next two weeks', 'missing_information': []}
--------------------------------------------------
2
Complete Enquiry
Hi, I'm David Wilson from Nova Manufacturing. We want to develop an AI-powered predictive maintenance system that can analyze machine sensor data and alert our maintenance team before equipment fails. We hope to begin the project next month.
{'client_name': 'David Wilson', 'company': 'Nova Manufacturing', 'industry': '', 'requirement': ['AI-powered predictive maintenance system', 'analyze machine sensor data', 'alert maintenan